In [4]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
warnings.filterwarnings("ignore", category=FutureWarning)

# Main project folder
PROJECT_ROOT = Path(
    "/Users/adewale/Documents/food_security_predictor"
)

# Exact raw FAOSTAT folder shown in Finder
RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "faostat_food_balance"
)

# Raw FAOSTAT files
FBS_RAW_PATH = (
    RAW_DATA_DIR
    / "FoodBalanceSheets_E_All_Data.csv"
)

FBS_NOFLAG_PATH = (
    RAW_DATA_DIR
    / "FoodBalanceSheets_E_All_Data_NOFLAG.csv"
)

AREA_CODES_PATH = (
    RAW_DATA_DIR
    / "FoodBalanceSheets_E_AreaCodes.csv"
)

ELEMENTS_PATH = (
    RAW_DATA_DIR
    / "FoodBalanceSheets_E_Elements.csv"
)

FLAGS_PATH = (
    RAW_DATA_DIR
    / "FoodBalanceSheets_E_Flags.csv"
)

ITEM_CODES_PATH = (
    RAW_DATA_DIR
    / "FoodBalanceSheets_E_ItemCodes.csv"
)

# Destination for the new Africa-first data
AFRICA_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "africa_first"
)

AFRICA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

YEAR_START = 2010
YEAR_END = 2023
EXPECTED_YEARS = list(
    range(YEAR_START, YEAR_END + 1)
)

CORE_ELEMENTS = [
    "Production",
    "Import quantity",
    "Domestic supply quantity",
    "Food",
    "Food supply quantity (kg/cap/yr)",
    "Food supply (kcal/cap/d)",
]

POPULATION_ELEMENT = (
    "Total Population - Both sexes"
)

raw_paths = {
    "Main data with flags": FBS_RAW_PATH,
    "Data without flags": FBS_NOFLAG_PATH,
    "Area codes": AREA_CODES_PATH,
    "Elements": ELEMENTS_PATH,
    "Flags": FLAGS_PATH,
    "Item codes": ITEM_CODES_PATH,
}

for description, file_path in raw_paths.items():
    print(
        f"{description}: "
        f"{'FOUND' if file_path.exists() else 'NOT FOUND'}"
    )
    print(f"  {file_path}")

Main data with flags: FOUND
  /Users/adewale/Documents/food_security_predictor/data/raw/faostat_food_balance/FoodBalanceSheets_E_All_Data.csv
Data without flags: FOUND
  /Users/adewale/Documents/food_security_predictor/data/raw/faostat_food_balance/FoodBalanceSheets_E_All_Data_NOFLAG.csv
Area codes: FOUND
  /Users/adewale/Documents/food_security_predictor/data/raw/faostat_food_balance/FoodBalanceSheets_E_AreaCodes.csv
Elements: FOUND
  /Users/adewale/Documents/food_security_predictor/data/raw/faostat_food_balance/FoodBalanceSheets_E_Elements.csv
Flags: FOUND
  /Users/adewale/Documents/food_security_predictor/data/raw/faostat_food_balance/FoodBalanceSheets_E_Flags.csv
Item codes: FOUND
  /Users/adewale/Documents/food_security_predictor/data/raw/faostat_food_balance/FoodBalanceSheets_E_ItemCodes.csv


In [5]:
raw_sample = pd.read_csv(
    FBS_RAW_PATH,
    nrows=5,
    encoding="utf-8-sig"
)

print("Sample shape:", raw_sample.shape)
print("\nRaw columns:")

for number, column in enumerate(
    raw_sample.columns,
    start=1
):
    print(f"{number}. {column}")

print("\nFirst five rows:")
display(raw_sample)

Sample shape: (5, 51)

Raw columns:
1. Area Code
2. Area Code (M49)
3. Area
4. Item Code
5. Item Code (FBS)
6. Item
7. Element Code
8. Element
9. Unit
10. Y2010
11. Y2010F
12. Y2010N
13. Y2011
14. Y2011F
15. Y2011N
16. Y2012
17. Y2012F
18. Y2012N
19. Y2013
20. Y2013F
21. Y2013N
22. Y2014
23. Y2014F
24. Y2014N
25. Y2015
26. Y2015F
27. Y2015N
28. Y2016
29. Y2016F
30. Y2016N
31. Y2017
32. Y2017F
33. Y2017N
34. Y2018
35. Y2018F
36. Y2018N
37. Y2019
38. Y2019F
39. Y2019N
40. Y2020
41. Y2020F
42. Y2020N
43. Y2021
44. Y2021F
45. Y2021N
46. Y2022
47. Y2022F
48. Y2022N
49. Y2023
50. Y2023F
51. Y2023N

First five rows:


,Area Code,Area Code (M49),Area,Item Code,Item Code (FBS),Item,Element Code,Element,Unit,Y2010,Y2010F,Y2010N,Y2011,Y2011F,Y2011N,Y2012,Y2012F,Y2012N,Y2013,Y2013F,Y2013N,Y2014,Y2014F,Y2014N,Y2015,Y2015F,Y2015N,Y2016,Y2016F,Y2016N,Y2017,Y2017F,Y2017N,Y2018,Y2018F,Y2018N,Y2019,Y2019F,Y2019N,Y2020,Y2020F,Y2020N,Y2021,Y2021F,Y2021N,Y2022,Y2022F,Y2022N,Y2023,Y2023F,Y2023N
0,2,'004,Afghanistan,2501,'S2501,Population,511,Total Population - Both sexes,1000 No,"28,284.09",X,NaN,"29,347.71",X,NaN,"30,560.03",X,NaN,"31,622.70",X,NaN,"32,792.52",X,NaN,"33,831.76",X,NaN,"34,700.61",X,NaN,"35,688.93",X,NaN,"36,743.04",X,NaN,"37,856.12",X,NaN,"39,068.98",X,NaN,"40,000.41",X,NaN,"40,578.84",X,NaN,"41,454.76",X,NaN
1,2,'004,Afghanistan,2901,'S2901,Grand Total,664,Food supply (kcal/capita/day),kcal/cap/d,"2,200.21",E,NaN,"2,171.86",E,NaN,"2,165.88",E,NaN,"2,205.33",E,NaN,"2,270.45",E,NaN,"2,251.27",E,NaN,"2,240.71",E,NaN,"2,307.21",E,NaN,"2,261.79",E,NaN,"2,223.10",E,NaN,"2,259.95",E,NaN,"2,244.73",E,NaN,"2,268.63",E,NaN,"2,314.63",E,NaN
2,2,'004,Afghanistan,2901,'S2901,Grand Total,661,Food supply (kcal),million Kcal,"22,714,261.33",E,NaN,"23,264,732.28",E,NaN,"24,159,167.28",E,NaN,"25,454,589.34",E,NaN,"27,175,651.99",E,NaN,"27,799,992.36",E,NaN,"28,380,157.97",E,NaN,"30,054,816.91",E,NaN,"30,333,352.71",E,NaN,"30,717,599.09",E,NaN,"32,227,343.15",E,NaN,"32,773,345.26",E,NaN,"33,601,287.96",E,NaN,"35,022,662.41",E,NaN
3,2,'004,Afghanistan,2901,'S2901,Grand Total,674,Protein supply quantity (g/capita/day),g/cap/d,65.54,E,NaN,63.74,E,NaN,62.67,E,NaN,63.24,E,NaN,65.55,E,NaN,63.93,E,NaN,64.82,E,NaN,65.33,E,NaN,62.84,E,NaN,60.84,E,NaN,63.21,E,NaN,61.36,E,NaN,61.55,E,NaN,62.58,E,NaN
4,2,'004,Afghanistan,2901,'S2901,Grand Total,671,Protein supply quantity (t),t,"676,666.08",E,NaN,"682,796.03",E,NaN,"699,046.85",E,NaN,"729,923.88",E,NaN,"784,625.74",E,NaN,"789,385.73",E,NaN,"820,992.96",E,NaN,"851,041.58",E,NaN,"842,749.06",E,NaN,"840,696.76",E,NaN,"901,388.20",E,NaN,"895,904.37",E,NaN,"911,681.15",E,NaN,"946,916.80",E,NaN


In [6]:
import re
from collections import Counter

identifier_dtypes = {
    "Area Code (M49)": "string",
    "Item Code (FBS)": "string",
}

fbs_raw = pd.read_csv(
    FBS_RAW_PATH,
    encoding="utf-8-sig",
    dtype=identifier_dtypes,
    low_memory=False
)

print("Raw FAO data loaded successfully.")
print(f"Rows: {len(fbs_raw):,}")
print(f"Columns: {fbs_raw.shape[1]:,}")

memory_mb = (
    fbs_raw.memory_usage(
        index=True,
        deep=True
    ).sum()
    / (1024 ** 2)
)

print(f"Memory usage: {memory_mb:,.2f} MB")

Raw FAO data loaded successfully.
Rows: 373,670
Columns: 51
Memory usage: 457.55 MB


In [7]:
value_columns = sorted(
    [
        column
        for column in fbs_raw.columns
        if re.fullmatch(r"Y\d{4}", column)
    ],
    key=lambda column: int(column[1:])
)

flag_columns = sorted(
    [
        column
        for column in fbs_raw.columns
        if re.fullmatch(r"Y\d{4}F", column)
    ],
    key=lambda column: int(column[1:5])
)

note_columns = sorted(
    [
        column
        for column in fbs_raw.columns
        if re.fullmatch(r"Y\d{4}N", column)
    ],
    key=lambda column: int(column[1:5])
)

detected_years = [
    int(column[1:])
    for column in value_columns
]

print("Value columns:", len(value_columns))
print("Flag columns:", len(flag_columns))
print("Note columns:", len(note_columns))
print("Detected years:", detected_years)

assert detected_years == EXPECTED_YEARS, (
    f"Unexpected years detected: {detected_years}"
)

assert len(value_columns) == len(flag_columns), (
    "The number of value and flag columns does not match."
)

assert len(value_columns) == len(note_columns), (
    "The number of value and note columns does not match."
)

print("\nYear-column structure validated successfully.")

Value columns: 14
Flag columns: 14
Note columns: 14
Detected years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

Year-column structure validated successfully.


In [8]:
identity_columns = [
    "Area Code",
    "Area Code (M49)",
    "Area",
    "Item Code",
    "Item Code (FBS)",
    "Item",
    "Element Code",
    "Element",
    "Unit",
]

duplicate_identity_rows = fbs_raw.duplicated(
    subset=identity_columns,
    keep=False
)

missing_value_cells = (
    fbs_raw[value_columns]
    .isna()
    .sum()
    .sum()
)

non_null_note_cells = (
    fbs_raw[note_columns]
    .notna()
    .sum()
    .sum()
)

flag_counter = Counter()

for column in flag_columns:
    flag_counter.update(
        fbs_raw[column]
        .dropna()
        .astype(str)
        .str.strip()
    )

source_summary = pd.Series(
    {
        "Rows": len(fbs_raw),
        "Columns": fbs_raw.shape[1],
        "Unique areas": fbs_raw["Area"].nunique(),
        "Unique items": fbs_raw["Item"].nunique(),
        "Unique item codes": fbs_raw["Item Code"].nunique(),
        "Unique FBS item codes": (
            fbs_raw["Item Code (FBS)"].nunique()
        ),
        "Unique elements": (
            fbs_raw["Element"].nunique()
        ),
        "Unique units": fbs_raw["Unit"].nunique(),
        "Duplicate identity rows": (
            duplicate_identity_rows.sum()
        ),
        "Missing value cells": missing_value_cells,
        "Non-null note cells": non_null_note_cells,
    },
    name="Result"
)

display(source_summary.to_frame())

print("\nFlag counts:")

flag_summary = pd.DataFrame(
    flag_counter.items(),
    columns=["Flag", "Count"]
).sort_values(
    "Count",
    ascending=False
)

flag_summary["Percentage"] = (
    flag_summary["Count"]
    / flag_summary["Count"].sum()
    * 100
)

display(flag_summary)

,Result
Rows,373670
Columns,51
Unique areas,213
Unique items,120
Unique item codes,123
Unique FBS item codes,123
Unique elements,21
Unique units,7
Duplicate identity rows,0
Missing value cells,410883



Flag counts:


,Flag,Count,Percentage
1,E,2618688,54.32
2,I,2198908,45.62
0,X,2901,0.06


In [9]:
# Normalise FAO M49 codes:
# examples such as "'004" become "004"

fbs_raw["M49 Code"] = (
    fbs_raw["Area Code (M49)"]
    .astype("string")
    .str.extract(r"(\d+)", expand=False)
    .str.zfill(3)
)

invalid_m49_rows = fbs_raw[
    fbs_raw["M49 Code"].isna()
]

print(
    f"Rows with invalid M49 codes: "
    f"{len(invalid_m49_rows):,}"
)

assert invalid_m49_rows.empty, (
    "Some M49 codes could not be normalised."
)

Rows with invalid M49 codes: 0


In [10]:
AFRICA_M49_REFERENCE = {
    "012": "Algeria",
    "024": "Angola",
    "204": "Benin",
    "072": "Botswana",
    "854": "Burkina Faso",
    "108": "Burundi",
    "132": "Cabo Verde",
    "120": "Cameroon",
    "140": "Central African Republic",
    "148": "Chad",
    "174": "Comoros",
    "178": "Congo",
    "180": "Democratic Republic of the Congo",
    "384": "Côte d'Ivoire",
    "262": "Djibouti",
    "818": "Egypt",
    "226": "Equatorial Guinea",
    "232": "Eritrea",
    "748": "Eswatini",
    "231": "Ethiopia",
    "266": "Gabon",
    "270": "Gambia",
    "288": "Ghana",
    "324": "Guinea",
    "624": "Guinea-Bissau",
    "404": "Kenya",
    "426": "Lesotho",
    "430": "Liberia",
    "434": "Libya",
    "450": "Madagascar",
    "454": "Malawi",
    "466": "Mali",
    "478": "Mauritania",
    "480": "Mauritius",
    "504": "Morocco",
    "508": "Mozambique",
    "516": "Namibia",
    "562": "Niger",
    "566": "Nigeria",
    "646": "Rwanda",
    "678": "Sao Tome and Principe",
    "686": "Senegal",
    "690": "Seychelles",
    "694": "Sierra Leone",
    "706": "Somalia",
    "710": "South Africa",
    "728": "South Sudan",
    "729": "Sudan",
    "834": "United Republic of Tanzania",
    "768": "Togo",
    "788": "Tunisia",
    "800": "Uganda",
    "894": "Zambia",
    "716": "Zimbabwe",
    "732": "Western Sahara",
}

africa_reference = pd.DataFrame(
    [
        {
            "M49 Code": m49_code,
            "Reference Area": country_name,
        }
        for m49_code, country_name
        in AFRICA_M49_REFERENCE.items()
    ]
)

# Western Sahara is kept visible for review,
# but is not included automatically.
africa_reference["Default Inclusion"] = (
    africa_reference["M49 Code"] != "732"
)

print(
    "African entities in reference:",
    len(africa_reference)
)

print(
    "Included by default:",
    africa_reference[
        "Default Inclusion"
    ].shape[0]
)

African entities in reference: 55
Included by default: 55


In [11]:
source_area_lookup = (
    fbs_raw[
        [
            "M49 Code",
            "Area Code",
            "Area",
        ]
    ]
    .drop_duplicates()
    .sort_values("M49 Code")
)

africa_area_review = (
    africa_reference.merge(
        source_area_lookup,
        on="M49 Code",
        how="left",
        validate="one_to_one"
    )
)

africa_area_review["Present in FAO"] = (
    africa_area_review["Area"].notna()
)

africa_area_review["Name Matches"] = (
    africa_area_review["Reference Area"]
    == africa_area_review["Area"]
)

africa_area_review["Include"] = (
    africa_area_review["Default Inclusion"]
    & africa_area_review["Present in FAO"]
)

print("African entity matching summary:\n")

display(
    africa_area_review[
        [
            "Present in FAO",
            "Default Inclusion",
            "Include",
        ]
    ]
    .value_counts()
    .rename("Number of entities")
    .reset_index()
)

print("\nMissing or name-mismatched entities:\n")

display(
    africa_area_review.loc[
        (~africa_area_review["Present in FAO"])
        | (~africa_area_review["Name Matches"]),
        [
            "M49 Code",
            "Reference Area",
            "Area",
            "Present in FAO",
            "Name Matches",
            "Default Inclusion",
            "Include",
        ]
    ]
)

African entity matching summary:



,Present in FAO,Default Inclusion,Include,Number of entities
0,True,True,True,43
1,False,True,False,11
2,False,False,False,1



Missing or name-mismatched entities:



,M49 Code,Reference Area,Area,Present in FAO,Name Matches,Default Inclusion,Include
2,204,Benin,NaN,False,False,True,False
5,108,Burundi,NaN,False,False,True,False
8,140,Central African Republic,NaN,False,False,True,False
9,148,Chad,NaN,False,False,True,False
16,226,Equatorial Guinea,NaN,False,False,True,False
17,232,Eritrea,NaN,False,False,True,False
31,466,Mali,NaN,False,False,True,False
44,706,Somalia,NaN,False,False,True,False
46,728,South Sudan,NaN,False,False,True,False
47,729,Sudan,NaN,False,False,True,False


In [12]:
included_africa_codes = set(
    africa_area_review.loc[
        africa_area_review["Include"],
        "M49 Code"
    ]
)

africa_raw_wide = fbs_raw.loc[
    fbs_raw["M49 Code"].isin(
        included_africa_codes
    )
].copy()

africa_scope_summary = pd.Series(
    {
        "Rows": len(africa_raw_wide),
        "Countries": (
            africa_raw_wide["Area"].nunique()
        ),
        "Items": (
            africa_raw_wide["Item"].nunique()
        ),
        "Item codes": (
            africa_raw_wide["Item Code"].nunique()
        ),
        "Elements": (
            africa_raw_wide["Element"].nunique()
        ),
        "Units": (
            africa_raw_wide["Unit"].nunique()
        ),
        "Earliest year": min(detected_years),
        "Latest year": max(detected_years),
    },
    name="Africa subset"
)

display(africa_scope_summary.to_frame())

,Africa subset
Rows,72769
Countries,43
Items,119
Item codes,122
Elements,21
Units,7
Earliest year,2010
Latest year,2023


In [13]:
print(
    africa_reference[
        "Default Inclusion"
    ].value_counts(dropna=False)
)

print(
    "\nNumber included by default:",
    africa_reference[
        "Default Inclusion"
    ].sum()
)

Default Inclusion
True     54
False     1
Name: count, dtype: int64

Number included by default: 54


In [14]:
missing_reference_countries = (
    africa_area_review.loc[
        ~africa_area_review["Present in FAO"],
        [
            "M49 Code",
            "Reference Area",
            "Default Inclusion",
        ]
    ]
    .copy()
)

source_area_lookup_clean = (
    source_area_lookup.copy()
)

source_area_lookup_clean["Area Clean"] = (
    source_area_lookup_clean["Area"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

missing_reference_countries["Reference Clean"] = (
    missing_reference_countries["Reference Area"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

name_based_check = (
    missing_reference_countries.merge(
        source_area_lookup_clean[
            [
                "M49 Code",
                "Area Code",
                "Area",
                "Area Clean",
            ]
        ],
        left_on="Reference Clean",
        right_on="Area Clean",
        how="left",
        suffixes=(
            " Expected",
            " Found",
        )
    )
)

print(
    "Name-based check in the main FAO data:"
)

display(
    name_based_check[
        [
            "Reference Area",
            "M49 Code Expected",
            "Area",
            "M49 Code Found",
            "Area Code",
            "Default Inclusion",
        ]
    ]
)

Name-based check in the main FAO data:


,Reference Area,M49 Code Expected,Area,M49 Code Found,Area Code,Default Inclusion
0,Benin,204,NaN,<NA>,NaN,True
1,Burundi,108,NaN,<NA>,NaN,True
2,Central African Republic,140,NaN,<NA>,NaN,True
3,Chad,148,NaN,<NA>,NaN,True
4,Equatorial Guinea,226,NaN,<NA>,NaN,True
5,Eritrea,232,NaN,<NA>,NaN,True
6,Mali,466,NaN,<NA>,NaN,True
7,Somalia,706,NaN,<NA>,NaN,True
8,South Sudan,728,NaN,<NA>,NaN,True
9,Sudan,729,NaN,<NA>,NaN,True


In [15]:
area_codes_raw = pd.read_csv(
    AREA_CODES_PATH,
    encoding="utf-8-sig",
    dtype="string"
)

area_codes_raw.columns = (
    area_codes_raw.columns
    .astype("string")
    .str.strip()
)

print("Area-code lookup shape:", area_codes_raw.shape)

print("\nArea-code lookup columns:")
print(area_codes_raw.columns.tolist())

display(area_codes_raw.head())

Area-code lookup shape: (426, 3)

Area-code lookup columns:
['Area Code', 'M49 Code', 'Area']


,Area Code,M49 Code,Area
0,2,'004,Afghanistan
1,2,'004,Afghanistan
2,5100,'002,Africa
3,5100,'002,Africa
4,3,'008,Albania


In [16]:
missing_country_names = (
    missing_reference_countries.loc[
        missing_reference_countries[
            "Default Inclusion"
        ],
        "Reference Area"
    ]
    .tolist()
)

missing_name_pattern = "|".join(
    re.escape(country)
    for country in missing_country_names
)

main_data_name_search = (
    fbs_raw.loc[
        fbs_raw["Area"]
        .astype("string")
        .str.contains(
            missing_name_pattern,
            case=False,
            na=False,
            regex=True
        ),
        [
            "Area Code",
            "Area Code (M49)",
            "M49 Code",
            "Area",
        ]
    ]
    .drop_duplicates()
    .sort_values("Area")
)

print(
    "Possible matches in the main data:"
)

display(main_data_name_search)

lookup_name_column = next(
    (
        column
        for column in area_codes_raw.columns
        if column.strip().casefold() == "area"
    ),
    None
)

if lookup_name_column is None:
    print(
        "No column named 'Area' was found "
        "in the area-code lookup."
    )
else:
    area_lookup_name_search = (
        area_codes_raw.loc[
            area_codes_raw[
                lookup_name_column
            ]
            .astype("string")
            .str.contains(
                missing_name_pattern,
                case=False,
                na=False,
                regex=True
            )
        ]
        .drop_duplicates()
    )

    print(
        "Possible matches in the "
        "area-code lookup:"
    )

    display(area_lookup_name_search)

Possible matches in the main data:


,Area Code,Area Code (M49),M49 Code,Area


Possible matches in the area-code lookup:


,Area Code,M49 Code,Area


In [17]:
africa_area_review["Exclusion Reason"] = ""

missing_country_mask = (
    africa_area_review["Default Inclusion"]
    & ~africa_area_review["Present in FAO"]
)

africa_area_review.loc[
    missing_country_mask,
    "Exclusion Reason"
] = (
    "No records in this FAOSTAT "
    "Food Balance Sheets release"
)

western_sahara_mask = (
    africa_area_review["M49 Code"] == "732"
)

africa_area_review.loc[
    western_sahara_mask,
    "Exclusion Reason"
] = (
    "Territorial review case; excluded "
    "from default sovereign-country scope "
    "and absent from source"
)

africa_area_review["Scope Status"] = np.where(
    africa_area_review["Include"],
    "Included",
    "Excluded"
)

print("Final geographic scope:\n")

display(
    africa_area_review[
        "Scope Status"
    ]
    .value_counts()
    .rename("Countries")
    .to_frame()
)

print("\nIncluded countries:\n")

display(
    africa_area_review.loc[
        africa_area_review["Include"],
        [
            "M49 Code",
            "Area Code",
            "Area",
        ]
    ]
    .sort_values("Area")
    .reset_index(drop=True)
)

print("\nExcluded African entities:\n")

display(
    africa_area_review.loc[
        ~africa_area_review["Include"],
        [
            "M49 Code",
            "Reference Area",
            "Present in FAO",
            "Exclusion Reason",
        ]
    ]
    .sort_values("Reference Area")
    .reset_index(drop=True)
)

Final geographic scope:



,Countries
Scope Status,
Included,43
Excluded,12



Included countries:



,M49 Code,Area Code,Area
0,012,4.00,Algeria
1,024,7.00,Angola
2,072,20.00,Botswana
3,854,233.00,Burkina Faso
4,132,35.00,Cabo Verde
5,120,32.00,Cameroon
6,174,45.00,Comoros
7,178,46.00,Congo
8,384,107.00,Côte d'Ivoire
9,180,250.00,Democratic Republic of the Congo



Excluded African entities:



,M49 Code,Reference Area,Present in FAO,Exclusion Reason
0,204,Benin,False,No records in this FAOSTAT Food Balance Sheets...
1,108,Burundi,False,No records in this FAOSTAT Food Balance Sheets...
2,140,Central African Republic,False,No records in this FAOSTAT Food Balance Sheets...
3,148,Chad,False,No records in this FAOSTAT Food Balance Sheets...
4,226,Equatorial Guinea,False,No records in this FAOSTAT Food Balance Sheets...
5,232,Eritrea,False,No records in this FAOSTAT Food Balance Sheets...
6,466,Mali,False,No records in this FAOSTAT Food Balance Sheets...
7,706,Somalia,False,No records in this FAOSTAT Food Balance Sheets...
8,728,South Sudan,False,No records in this FAOSTAT Food Balance Sheets...
9,729,Sudan,False,No records in this FAOSTAT Food Balance Sheets...


In [18]:
long_identifier_columns = [
    "Area Code",
    "Area Code (M49)",
    "M49 Code",
    "Area",
    "Item Code",
    "Item Code (FBS)",
    "Item",
    "Element Code",
    "Element",
    "Unit",
]

africa_year_frames = []

for year in EXPECTED_YEARS:
    value_column = f"Y{year}"
    flag_column = f"Y{year}F"

    year_frame = (
        africa_raw_wide[
            long_identifier_columns
        ]
        .copy()
    )

    year_frame["Year"] = year
    year_frame["Value"] = (
        africa_raw_wide[
            value_column
        ].to_numpy()
    )
    year_frame["Flag"] = (
        africa_raw_wide[
            flag_column
        ]
        .astype("string")
        .str.strip()
        .to_numpy()
    )

    africa_year_frames.append(
        year_frame
    )

africa_long = pd.concat(
    africa_year_frames,
    ignore_index=True
)

del africa_year_frames

print("Africa long table created.")
print(f"Rows: {len(africa_long):,}")
print(f"Columns: {africa_long.shape[1]:,}")

Africa long table created.
Rows: 1,018,766
Columns: 13


In [19]:
expected_long_rows = (
    len(africa_raw_wide)
    * len(EXPECTED_YEARS)
)

duplicate_long_keys = (
    africa_long.duplicated(
        subset=(
            long_identifier_columns
            + ["Year"]
        ),
        keep=False
    )
    .sum()
)

values_without_flags = (
    africa_long["Value"].notna()
    & africa_long["Flag"].isna()
).sum()

flags_without_values = (
    africa_long["Value"].isna()
    & africa_long["Flag"].notna()
).sum()

long_summary = pd.Series(
    {
        "Expected rows": expected_long_rows,
        "Actual rows": len(africa_long),
        "Countries": (
            africa_long["Area"].nunique()
        ),
        "Items": (
            africa_long["Item"].nunique()
        ),
        "Item codes": (
            africa_long["Item Code"].nunique()
        ),
        "Elements": (
            africa_long["Element"].nunique()
        ),
        "Years": (
            africa_long["Year"].nunique()
        ),
        "Recorded values": (
            africa_long["Value"].notna().sum()
        ),
        "Missing values": (
            africa_long["Value"].isna().sum()
        ),
        "Duplicate long keys": (
            duplicate_long_keys
        ),
        "Values without flags": (
            values_without_flags
        ),
        "Flags without values": (
            flags_without_values
        ),
    },
    name="Result"
)

display(long_summary.to_frame())

assert len(africa_long) == expected_long_rows
assert duplicate_long_keys == 0
assert values_without_flags == 0
assert flags_without_values == 0

print(
    "\nLong transformation validated successfully."
)

,Result
Expected rows,1018766
Actual rows,1018766
Countries,43
Items,119
Item codes,122
Elements,21
Years,14
Recorded values,945584
Missing values,73182
Duplicate long keys,0



Long transformation validated successfully.


In [20]:
annual_coverage = (
    africa_long.groupby("Year")
    .agg(
        Total_rows=("Value", "size"),
        Recorded_values=("Value", "count"),
    )
    .reset_index()
)

annual_coverage["Missing_values"] = (
    annual_coverage["Total_rows"]
    - annual_coverage["Recorded_values"]
)

annual_coverage["Coverage_percentage"] = (
    annual_coverage["Recorded_values"]
    / annual_coverage["Total_rows"]
    * 100
)

print("Annual coverage:")
display(annual_coverage)

africa_flag_summary = (
    africa_long.loc[
        africa_long["Value"].notna()
    ]
    .groupby("Flag", dropna=False)
    .size()
    .rename("Count")
    .reset_index()
)

africa_flag_summary["Percentage"] = (
    africa_flag_summary["Count"]
    / africa_flag_summary["Count"].sum()
    * 100
)

print("\nAfrican quality flags:")
display(
    africa_flag_summary.sort_values(
        "Count",
        ascending=False
    )
)

Annual coverage:


,Year,Total_rows,Recorded_values,Missing_values,Coverage_percentage
0,2010,72769,67080,5689,92.18
1,2011,72769,67135,5634,92.26
2,2012,72769,67290,5479,92.47
3,2013,72769,67314,5455,92.50
4,2014,72769,67312,5457,92.50
5,2015,72769,67301,5468,92.49
6,2016,72769,67401,5368,92.62
7,2017,72769,67648,5121,92.96
8,2018,72769,67564,5205,92.85
9,2019,72769,67658,5111,92.98



African quality flags:


,Flag,Count,Percentage
1,I,533085,56.38
0,E,411897,43.56
2,X,602,0.06


In [21]:
population_long = (
    africa_long.loc[
        africa_long["Element"].eq(
            POPULATION_ELEMENT
        )
    ]
    .copy()
)

commodity_long = (
    africa_long.loc[
        ~africa_long["Element"].eq(
            POPULATION_ELEMENT
        )
    ]
    .copy()
)

population_key_counts = (
    population_long.groupby(
        [
            "M49 Code",
            "Area",
            "Year",
        ],
        dropna=False
    )
    .size()
    .rename("Row count")
    .reset_index()
)

duplicate_population_keys = (
    population_key_counts.loc[
        population_key_counts[
            "Row count"
        ].gt(1)
    ]
)

population_summary = pd.Series(
    {
        "Population rows": (
            len(population_long)
        ),
        "Expected population rows": (
            43 * len(EXPECTED_YEARS)
        ),
        "Countries": (
            population_long[
                "Area"
            ].nunique()
        ),
        "Years": (
            population_long[
                "Year"
            ].nunique()
        ),
        "Recorded population values": (
            population_long[
                "Value"
            ].notna().sum()
        ),
        "Missing population values": (
            population_long[
                "Value"
            ].isna().sum()
        ),
        "Duplicate country-year keys": (
            len(duplicate_population_keys)
        ),
        "Population items": (
            population_long[
                "Item"
            ].nunique()
        ),
        "Population units": (
            population_long[
                "Unit"
            ].nunique()
        ),
        "Population flags": (
            population_long[
                "Flag"
            ].nunique()
        ),
        "Remaining commodity rows": (
            len(commodity_long)
        ),
    },
    name="Result"
)

display(population_summary.to_frame())

print("\nPopulation item, element, unit and flag:")

display(
    population_long[
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
            "Element Code",
            "Element",
            "Unit",
            "Flag",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Item",
            "Element",
            "Flag",
        ]
    )
)

assert len(population_long) == (
    43 * len(EXPECTED_YEARS)
)

assert population_long["Value"].notna().all()

assert duplicate_population_keys.empty

print(
    "\nPopulation separation "
    "validated successfully."
)

,Result
Population rows,602
Expected population rows,602
Countries,43
Years,14
Recorded population values,602
Missing population values,0
Duplicate country-year keys,0
Population items,1
Population units,1
Population flags,1



Population item, element, unit and flag:


,Item Code,Item Code (FBS),Item,Element Code,Element,Unit,Flag
0,2501,'S2501,Population,511,Total Population - Both sexes,1000 No,X



Population separation validated successfully.


In [22]:
commodity_flag_summary = (
    commodity_long.loc[
        commodity_long["Value"].notna()
    ]
    .groupby("Flag", dropna=False)
    .size()
    .rename("Count")
    .reset_index()
)

commodity_flag_summary["Percentage"] = (
    commodity_flag_summary["Count"]
    / commodity_flag_summary["Count"].sum()
    * 100
)

print("Commodity-only quality flags:")

display(
    commodity_flag_summary.sort_values(
        "Count",
        ascending=False
    )
)

Commodity-only quality flags:


,Flag,Count,Percentage
1,I,533085,56.41
0,E,411897,43.59


In [23]:
element_unit_audit = (
    commodity_long.groupby(
        [
            "Element Code",
            "Element",
            "Unit",
        ],
        dropna=False
    )
    .agg(
        Source_rows=(
            "Value",
            "size"
        ),
        Recorded_values=(
            "Value",
            "count"
        ),
        Countries=(
            "M49 Code",
            lambda values: (
                commodity_long.loc[
                    values.index,
                    ["M49 Code", "Value"]
                ]
                .dropna(subset=["Value"])
                ["M49 Code"]
                .nunique()
            )
        ),
        Items=(
            "Item",
            "nunique"
        ),
        First_year=(
            "Year",
            "min"
        ),
        Last_year=(
            "Year",
            "max"
        ),
    )
    .reset_index()
)

element_unit_audit[
    "Missing_values"
] = (
    element_unit_audit["Source_rows"]
    - element_unit_audit["Recorded_values"]
)

element_unit_audit[
    "Coverage_percentage"
] = (
    element_unit_audit["Recorded_values"]
    / element_unit_audit["Source_rows"]
    * 100
)

element_unit_audit = (
    element_unit_audit.sort_values(
        [
            "Element",
            "Unit",
        ]
    )
    .reset_index(drop=True)
)

display(element_unit_audit)

,Element Code,Element,Unit,Source_rows,Recorded_values,Countries,Items,First_year,Last_year,Missing_values,Coverage_percentage
0,5301,Domestic supply quantity,1000 t,70098,67799,43,115,2010,2023,2299,96.72
1,5911,Export quantity,1000 t,63140,47575,43,115,2010,2023,15565,75.35
2,684,Fat supply quantity (g/capita/day),g/cap/d,66570,64273,43,115,2010,2023,2297,96.55
3,681,Fat supply quantity (t),t,66570,64273,43,115,2010,2023,2297,96.55
4,5521,Feed,1000 t,23996,18288,43,68,2010,2023,5708,76.21
5,5142,Food,1000 t,66696,64276,43,115,2010,2023,2420,96.37
6,661,Food supply (kcal),million Kcal,66570,64273,43,115,2010,2023,2297,96.55
7,664,Food supply (kcal/capita/day),kcal/cap/d,66570,64273,43,115,2010,2023,2297,96.55
8,645,Food supply quantity (kg/capita/yr),kg/cap,66696,64276,43,115,2010,2023,2420,96.37
9,5611,Import quantity,1000 t,69244,64627,43,115,2010,2023,4617,93.33


In [24]:
CORE_ELEMENTS = [
    "Production",
    "Import quantity",
    "Domestic supply quantity",
    "Food",
    "Food supply quantity (kg/capita/yr)",
    "Food supply (kcal/capita/day)",
]

available_elements = set(
    commodity_long["Element"]
    .dropna()
    .unique()
)

missing_core_elements = [
    element
    for element in CORE_ELEMENTS
    if element not in available_elements
]

print("Core elements:")
for element in CORE_ELEMENTS:
    print(f" - {element}")

print(
    "\nMissing core elements:",
    missing_core_elements
)

assert not missing_core_elements, (
    f"Core elements not found: "
    f"{missing_core_elements}"
)

core_element_audit = (
    element_unit_audit.loc[
        element_unit_audit[
            "Element"
        ].isin(CORE_ELEMENTS)
    ]
    .copy()
    .sort_values("Element")
)

display(core_element_audit)

Core elements:
 - Production
 - Import quantity
 - Domestic supply quantity
 - Food
 - Food supply quantity (kg/capita/yr)
 - Food supply (kcal/capita/day)

Missing core elements: []


,Element Code,Element,Unit,Source_rows,Recorded_values,Countries,Items,First_year,Last_year,Missing_values,Coverage_percentage
0,5301,Domestic supply quantity,1000 t,70098,67799,43,115,2010,2023,2299,96.72
5,5142,Food,1000 t,66696,64276,43,115,2010,2023,2420,96.37
7,664,Food supply (kcal/capita/day),kcal/cap/d,66570,64273,43,115,2010,2023,2297,96.55
8,645,Food supply quantity (kg/capita/yr),kg/cap,66696,64276,43,115,2010,2023,2420,96.37
9,5611,Import quantity,1000 t,69244,64627,43,115,2010,2023,4617,93.33
13,5511,Production,1000 t,46802,42965,43,114,2010,2023,3837,91.80


In [25]:
item_identity = (
    commodity_long[
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Item",
            "Item Code",
            "Item Code (FBS)",
        ]
    )
    .reset_index(drop=True)
)

item_identity_summary = pd.Series(
    {
        "Commodity item labels": (
            item_identity["Item"].nunique()
        ),
        "Standard item codes": (
            item_identity[
                "Item Code"
            ].nunique()
        ),
        "FBS item codes": (
            item_identity[
                "Item Code (FBS)"
            ].nunique()
        ),
        "Unique identity combinations": (
            len(item_identity)
        ),
    },
    name="Result"
)

display(item_identity_summary.to_frame())

,Result
Commodity item labels,118
Standard item codes,121
FBS item codes,121
Unique identity combinations,121


In [26]:
label_code_counts = (
    item_identity.groupby(
        "Item",
        dropna=False
    )
    .agg(
        Standard_code_count=(
            "Item Code",
            "nunique"
        ),
        FBS_code_count=(
            "Item Code (FBS)",
            "nunique"
        ),
    )
    .reset_index()
)

colliding_item_labels = (
    label_code_counts.loc[
        (
            label_code_counts[
                "Standard_code_count"
            ].gt(1)
        )
        | (
            label_code_counts[
                "FBS_code_count"
            ].gt(1)
        )
    ]
    .sort_values("Item")
)

print(
    "Item labels associated with "
    "multiple codes:"
)

display(colliding_item_labels)

collision_details = (
    item_identity.merge(
        colliding_item_labels[
            ["Item"]
        ],
        on="Item",
        how="inner",
        validate="many_to_one"
    )
    .sort_values(
        [
            "Item",
            "Item Code",
        ]
    )
    .reset_index(drop=True)
)

print("\nCollision details:")

display(collision_details)

Item labels associated with multiple codes:


,Item,Standard_code_count,FBS_code_count
32,Eggs,2,2
53,Milk - Excluding Butter,2,2
55,Miscellaneous,2,2



Collision details:


,Item Code,Item Code (FBS),Item
0,2744,'S2744,Eggs
1,2949,'S2949,Eggs
2,2848,'S2848,Milk - Excluding Butter
3,2948,'S2948,Milk - Excluding Butter
4,2899,'S2899,Miscellaneous
5,2928,'S2928,Miscellaneous


In [27]:
standard_code_label_counts = (
    item_identity.groupby(
        "Item Code",
        dropna=False
    )["Item"]
    .nunique()
)

fbs_code_label_counts = (
    item_identity.groupby(
        "Item Code (FBS)",
        dropna=False
    )["Item"]
    .nunique()
)

standard_code_label_collisions = (
    standard_code_label_counts.loc[
        standard_code_label_counts.gt(1)
    ]
)

fbs_code_label_collisions = (
    fbs_code_label_counts.loc[
        fbs_code_label_counts.gt(1)
    ]
)

print(
    "Standard codes linked to "
    "multiple labels:",
    len(standard_code_label_collisions)
)

print(
    "FBS codes linked to "
    "multiple labels:",
    len(fbs_code_label_collisions)
)

Standard codes linked to multiple labels: 0
FBS codes linked to multiple labels: 0


In [28]:
staple_keywords = (
    r"rice|maize|wheat|cassava|yam|"
    r"millet|sorghum|potato|plantain|"
    r"pulse|bean|cow pea|cowpea"
)

candidate_item_identities = (
    item_identity.loc[
        item_identity["Item"]
        .astype("string")
        .str.contains(
            staple_keywords,
            case=False,
            regex=True,
            na=False
        )
    ]
    .sort_values(
        [
            "Item",
            "Item Code",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Potential staple item identities:"
)

display(candidate_item_identities)

Potential staple item identities:


,Item Code,Item Code (FBS),Item
0,2546,'S2546,Beans
1,2532,'S2532,Cassava and products
2,2633,'S2633,Cocoa Beans and products
3,2582,'S2582,Maize Germ Oil
4,2514,'S2514,Maize and products
5,2517,'S2517,Millet and products
6,2616,'S2616,Plantains
7,2531,'S2531,Potatoes and products
8,2911,'S2911,Pulses
9,2549,'S2549,"Pulses, Other and products"


In [29]:
ITEM_ID_COLUMNS = [
    "Item Code",
    "Item Code (FBS)",
    "Item",
]

NUMBER_OF_COUNTRIES = (
    commodity_long["M49 Code"].nunique()
)

NUMBER_OF_YEARS = (
    commodity_long["Year"].nunique()
)

POSSIBLE_COUNTRY_YEARS = (
    NUMBER_OF_COUNTRIES
    * NUMBER_OF_YEARS
)

print(
    "Possible country-year observations "
    f"per item-element: "
    f"{POSSIBLE_COUNTRY_YEARS:,}"
)

core_observed = (
    commodity_long.loc[
        commodity_long["Element"].isin(
            CORE_ELEMENTS
        )
        & commodity_long["Value"].notna()
    ]
    .copy()
)

core_coverage_long = (
    core_observed.groupby(
        ITEM_ID_COLUMNS + ["Element"],
        dropna=False
    )
    .size()
    .rename("Recorded country-years")
    .reset_index()
)

core_coverage_long[
    "Coverage percentage"
] = (
    core_coverage_long[
        "Recorded country-years"
    ]
    / POSSIBLE_COUNTRY_YEARS
    * 100
)

core_coverage_wide = (
    core_coverage_long.pivot(
        index=ITEM_ID_COLUMNS,
        columns="Element",
        values="Coverage percentage"
    )
    .reset_index()
)

core_coverage_wide.columns.name = None

coverage_column_names = {
    "Production":
        "Production coverage %",
    "Import quantity":
        "Import coverage %",
    "Domestic supply quantity":
        "Domestic supply coverage %",
    "Food":
        "Food coverage %",
    "Food supply quantity (kg/capita/yr)":
        "Per-capita quantity coverage %",
    "Food supply (kcal/capita/day)":
        "Calorie coverage %",
}

core_coverage_wide = (
    core_coverage_wide.rename(
        columns=coverage_column_names
    )
)

coverage_columns = list(
    coverage_column_names.values()
)

commodity_coverage = (
    item_identity.merge(
        core_coverage_wide,
        on=ITEM_ID_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

commodity_coverage[
    coverage_columns
] = (
    commodity_coverage[
        coverage_columns
    ]
    .fillna(0)
)

commodity_coverage[
    "Minimum core coverage %"
] = (
    commodity_coverage[
        coverage_columns
    ]
    .min(axis=1)
)

commodity_coverage[
    "Mean core coverage %"
] = (
    commodity_coverage[
        coverage_columns
    ]
    .mean(axis=1)
)

print(
    "Commodity coverage table created:",
    commodity_coverage.shape
)

Possible country-year observations per item-element: 602
Commodity coverage table created: (121, 11)


In [30]:
population_panel = (
    population_long[
        [
            "M49 Code",
            "Area",
            "Year",
            "Value",
            "Flag",
        ]
    ]
    .rename(
        columns={
            "Value": "Population 1000",
            "Flag": "Population flag",
        }
    )
    .copy()
)

assert not population_panel.duplicated(
    subset=[
        "M49 Code",
        "Year",
    ]
).any()

In [31]:
IMPORTANCE_YEARS = [
    2021,
    2022,
    2023,
]

calorie_element = (
    "Food supply (kcal/capita/day)"
)

recent_calories = (
    commodity_long.loc[
        commodity_long[
            "Element"
        ].eq(calorie_element)
        & commodity_long[
            "Year"
        ].isin(IMPORTANCE_YEARS)
        & commodity_long[
            "Value"
        ].notna(),
        ITEM_ID_COLUMNS
        + [
            "M49 Code",
            "Area",
            "Year",
            "Value",
            "Flag",
        ]
    ]
    .copy()
)

recent_calories = (
    recent_calories.merge(
        population_panel[
            [
                "M49 Code",
                "Year",
                "Population 1000",
            ]
        ],
        on=[
            "M49 Code",
            "Year",
        ],
        how="left",
        validate="many_to_one"
    )
)

assert recent_calories[
    "Population 1000"
].notna().all()

recent_calories[
    "Population-weighted calories"
] = (
    recent_calories["Value"]
    * recent_calories["Population 1000"]
)

recent_calories[
    "Positive calorie supply"
] = (
    recent_calories["Value"].gt(0)
)

In [32]:
calorie_importance = (
    recent_calories.groupby(
        ITEM_ID_COLUMNS,
        dropna=False
    )
    .agg(
        Recent_recorded_country_years=(
            "Value",
            "count"
        ),
        Observed_population_1000=(
            "Population 1000",
            "sum"
        ),
        Weighted_calorie_total=(
            "Population-weighted calories",
            "sum"
        ),
    )
    .reset_index()
)

calorie_importance[
    "Recent population-weighted kcal/capita/day"
] = (
    calorie_importance[
        "Weighted_calorie_total"
    ]
    / calorie_importance[
        "Observed_population_1000"
    ]
)

positive_country_counts = (
    recent_calories.loc[
        recent_calories[
            "Positive calorie supply"
        ]
    ]
    .groupby(
        ITEM_ID_COLUMNS,
        dropna=False
    )["M49 Code"]
    .nunique()
    .rename(
        "Countries with positive calories"
    )
    .reset_index()
)

country_average_calories = (
    recent_calories.groupby(
        ITEM_ID_COLUMNS
        + [
            "M49 Code",
            "Area",
        ],
        dropna=False
    )["Value"]
    .mean()
    .rename(
        "Country average kcal/capita/day"
    )
    .reset_index()
)

median_country_calories = (
    country_average_calories.groupby(
        ITEM_ID_COLUMNS,
        dropna=False
    )[
        "Country average kcal/capita/day"
    ]
    .median()
    .rename(
        "Median country kcal/capita/day"
    )
    .reset_index()
)

calorie_importance = (
    calorie_importance
    .merge(
        positive_country_counts,
        on=ITEM_ID_COLUMNS,
        how="left"
    )
    .merge(
        median_country_calories,
        on=ITEM_ID_COLUMNS,
        how="left"
    )
)

In [35]:
core_quality = core_observed.copy()

core_quality["Is imputed"] = (
    core_quality["Flag"].eq("I")
)

core_quality["Is estimated"] = (
    core_quality["Flag"].eq("E")
)

commodity_quality = (
    core_quality.groupby(
        ITEM_ID_COLUMNS,
        dropna=False
    )
    .agg(
        Core_recorded_values=(
            "Value",
            "count"
        ),
        Imputed_share=(
            "Is imputed",
            "mean"
        ),
        Estimated_share=(
            "Is estimated",
            "mean"
        ),
    )
    .reset_index()
)

commodity_quality[
    "Imputed percentage"
] = (
    commodity_quality[
        "Imputed_share"
    ]
    * 100
)

commodity_quality[
    "Estimated percentage"
] = (
    commodity_quality[
        "Estimated_share"
    ]
    * 100
)

commodity_quality = (
    commodity_quality.rename(
        columns={
            "Core_recorded_values":
                "Core recorded values"
        }
    )
    .drop(
        columns=[
            "Imputed_share",
            "Estimated_share",
        ]
    )
)

display(commodity_quality.head())

print(
    "Commodity quality table shape:",
    commodity_quality.shape
)

,Item Code,Item Code (FBS),Item,Core recorded values,Imputed percentage,Estimated percentage
0,2511,'S2511,Wheat and products,3386,64.44,35.56
1,2513,'S2513,Barley and products,3190,63.26,36.74
2,2514,'S2514,Maize and products,3570,66.27,33.73
3,2515,'S2515,Rye and products,2426,59.03,40.97
4,2516,'S2516,Oats,3077,61.85,38.15


Commodity quality table shape: (121, 6)


In [36]:
commodity_selection_evidence = (
    commodity_coverage
    .merge(
        calorie_importance,
        on=ITEM_ID_COLUMNS,
        how="left",
        validate="one_to_one"
    )
    .merge(
        commodity_quality,
        on=ITEM_ID_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

top_calorie_items = (
    commodity_selection_evidence[
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
            "Production coverage %",
            "Import coverage %",
            "Domestic supply coverage %",
            "Food coverage %",
            "Calorie coverage %",
            "Minimum core coverage %",
            "Recent population-weighted kcal/capita/day",
            "Median country kcal/capita/day",
            "Countries with positive calories",
            "Imputed percentage",
        ]
    ]
    .sort_values(
        "Recent population-weighted kcal/capita/day",
        ascending=False
    )
    .head(30)
    .reset_index(drop=True)
)

print("Top 30 items by recent calorie importance:")

display(top_calorie_items)

Top 30 items by recent calorie importance:


,Item Code,Item Code (FBS),Item,Production coverage %,Import coverage %,Domestic supply coverage %,Food coverage %,Calorie coverage %,Minimum core coverage %,Recent population-weighted kcal/capita/day,Median country kcal/capita/day,Countries with positive calories,Imputed percentage
0,2901,'S2901,Grand Total,0.00,0.00,0.00,0.00,100.00,0.00,"2,601.85","2,559.28",43.00,0.00
1,2903,'S2903,Vegetal Products,0.00,0.00,0.00,0.00,100.00,0.00,"2,416.91","2,323.62",43.00,0.00
2,2905,'S2905,Cereals - Excluding Beer,97.67,100.00,100.00,100.00,100.00,97.67,"1,187.50","1,173.35",43.00,0.00
3,2907,'S2907,Starchy Roots,97.67,100.00,100.00,100.00,100.00,97.67,437.34,168.31,43.00,0.00
4,2514,'S2514,Maize and products,93.02,100.00,100.00,100.00,100.00,93.02,387.69,272.05,43.00,66.27
5,2511,'S2511,Wheat and products,62.46,100.00,100.00,100.00,100.00,62.46,364.51,225.92,43.00,64.44
6,2532,'S2532,Cassava and products,69.77,91.53,98.34,95.85,95.85,69.77,302.68,68.51,38.00,64.97
7,2914,'S2914,Vegetable Oils,96.35,100.00,100.00,100.00,100.00,96.35,248.98,234.75,43.00,0.00
8,2807,'S2807,Rice and products,79.24,100.00,100.00,100.00,100.00,79.24,236.00,273.17,43.00,65.47
9,2941,'S2941,Animal Products,0.00,0.00,0.00,0.00,100.00,0.00,184.94,216.55,43.00,0.00


In [37]:
STAPLE_CANDIDATE_CODES = {
    "2511": "Wheat and products",
    "2514": "Maize and products",
    "2517": "Millet and products",
    "2518": "Sorghum and products",
    "2531": "Potatoes and products",
    "2532": "Cassava and products",
    "2533": "Sweet potatoes",
    "2535": "Yams",
    "2546": "Beans",
    "2549": "Pulses, Other and products",
    "2555": "Soyabeans",
    "2556": "Groundnuts",
    "2616": "Plantains",
    "2807": "Rice and products",
}

candidate_selection_table = (
    commodity_selection_evidence.loc[
        commodity_selection_evidence[
            "Item Code"
        ]
        .astype("string")
        .isin(STAPLE_CANDIDATE_CODES)
    ]
    [
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
            "Production coverage %",
            "Import coverage %",
            "Domestic supply coverage %",
            "Food coverage %",
            "Per-capita quantity coverage %",
            "Calorie coverage %",
            "Minimum core coverage %",
            "Mean core coverage %",
            "Recent population-weighted kcal/capita/day",
            "Median country kcal/capita/day",
            "Countries with positive calories",
            "Imputed percentage",
            "Estimated percentage",
        ]
    ]
    .sort_values(
        "Recent population-weighted kcal/capita/day",
        ascending=False
    )
    .reset_index(drop=True)
)

display(candidate_selection_table)

,Item Code,Item Code (FBS),Item,Production coverage %,Import coverage %,Domestic supply coverage %,Food coverage %,Per-capita quantity coverage %,Calorie coverage %,Minimum core coverage %,Mean core coverage %,Recent population-weighted kcal/capita/day,Median country kcal/capita/day,Countries with positive calories,Imputed percentage,Estimated percentage
0,2514,'S2514,Maize and products,93.02,100.00,100.00,100.00,100.00,100.00,93.02,98.84,387.69,272.05,43.00,66.27,33.73
1,2511,'S2511,Wheat and products,62.46,100.00,100.00,100.00,100.00,100.00,62.46,93.74,364.51,225.92,43.00,64.44,35.56
2,2532,'S2532,Cassava and products,69.77,91.53,98.34,95.85,95.85,95.85,69.77,91.20,302.68,68.51,38.00,64.97,35.03
3,2807,'S2807,Rice and products,79.24,100.00,100.00,100.00,100.00,100.00,79.24,96.54,236.00,273.17,43.00,65.47,34.53
4,2518,'S2518,Sorghum and products,76.74,78.90,89.04,82.89,82.89,82.89,76.74,82.23,92.87,16.99,37.00,66.40,33.60
5,2535,'S2535,Yams,41.20,42.03,68.60,61.30,61.30,61.63,41.20,56.01,83.14,1.38,21.00,63.42,36.58
6,2549,'S2549,"Pulses, Other and products",87.38,99.34,99.34,99.34,99.34,99.34,87.38,97.34,62.30,29.54,43.00,65.98,34.02
7,2616,'S2616,Plantains,38.87,57.81,82.89,78.57,78.57,78.57,38.87,69.21,44.11,0.76,28.00,62.16,37.84
8,2517,'S2517,Millet and products,67.44,83.22,92.36,76.41,76.41,76.41,67.44,78.71,42.82,5.81,34.00,67.64,32.36
9,2546,'S2546,Beans,63.46,98.34,99.34,99.34,99.34,99.34,63.46,93.19,32.41,10.24,41.00,64.47,35.53


In [38]:
commodity_selection_evidence[
    "Item Code numeric"
] = pd.to_numeric(
    commodity_selection_evidence[
        "Item Code"
    ],
    errors="coerce"
)

assert commodity_selection_evidence[
    "Item Code numeric"
].notna().all()

commodity_selection_evidence[
    "Item classification"
] = np.where(
    commodity_selection_evidence[
        "Item Code numeric"
    ].between(2900, 2999),
    "Aggregate category",
    "Individual commodity"
)

aggregate_item_review = (
    commodity_selection_evidence.loc[
        commodity_selection_evidence[
            "Item classification"
        ].eq("Aggregate category"),
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
            "Recent population-weighted kcal/capita/day",
            "Countries with positive calories",
        ]
    ]
    .sort_values("Item Code")
    .reset_index(drop=True)
)

print("FAO aggregate categories:")

display(aggregate_item_review)

print(
    "\nNumber of aggregate identities:",
    len(aggregate_item_review)
)

FAO aggregate categories:


,Item Code,Item Code (FBS),Item,Recent population-weighted kcal/capita/day,Countries with positive calories
0,2901,'S2901,Grand Total,"2,601.85",43.00
1,2903,'S2903,Vegetal Products,"2,416.91",43.00
2,2905,'S2905,Cereals - Excluding Beer,"1,187.50",43.00
3,2907,'S2907,Starchy Roots,437.34,43.00
4,2908,'S2908,Sugar Crops,10.15,18.00
5,2909,'S2909,Sugar & Sweeteners,140.22,43.00
6,2911,'S2911,Pulses,99.87,43.00
7,2912,'S2912,Treenuts,5.63,43.00
8,2913,'S2913,Oilcrops,68.89,43.00
9,2914,'S2914,Vegetable Oils,248.98,43.00



Number of aggregate identities: 24


In [39]:
individual_commodity_evidence = (
    commodity_selection_evidence.loc[
        commodity_selection_evidence[
            "Item classification"
        ].eq("Individual commodity")
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Individual commodity identities:",
    len(individual_commodity_evidence)
)

print(
    "Aggregate identities excluded:",
    len(aggregate_item_review)
)

assert "Grand Total" not in set(
    individual_commodity_evidence["Item"]
)

Individual commodity identities: 97
Aggregate identities excluded: 24


In [40]:
commodity_long[
    "Item Code numeric"
] = pd.to_numeric(
    commodity_long["Item Code"],
    errors="coerce"
)

individual_commodity_long = (
    commodity_long.loc[
        ~commodity_long[
            "Item Code numeric"
        ].between(2900, 2999)
    ]
    .copy()
)

print(
    f"Non-population rows before "
    f"aggregate exclusion: "
    f"{len(commodity_long):,}"
)

print(
    f"Individual-commodity rows: "
    f"{len(individual_commodity_long):,}"
)

print(
    "Individual commodity labels:",
    individual_commodity_long[
        "Item"
    ].nunique()
)

print(
    "Individual commodity codes:",
    individual_commodity_long[
        "Item Code"
    ].nunique()
)

Non-population rows before aggregate exclusion: 1,018,164
Individual-commodity rows: 808,584
Individual commodity labels: 97
Individual commodity codes: 97


In [41]:
STAPLE_CANDIDATE_CODES = {
    "2511",  # Wheat and products
    "2514",  # Maize and products
    "2517",  # Millet and products
    "2518",  # Sorghum and products
    "2531",  # Potatoes and products
    "2532",  # Cassava and products
    "2533",  # Sweet potatoes
    "2535",  # Yams
    "2546",  # Beans
    "2549",  # Pulses, Other and products
    "2552",  # Groundnuts
    "2555",  # Soyabeans
    "2616",  # Plantains
    "2807",  # Rice and products
}

candidate_selection_table = (
    individual_commodity_evidence.loc[
        individual_commodity_evidence[
            "Item Code"
        ]
        .astype("string")
        .isin(STAPLE_CANDIDATE_CODES)
    ]
    [
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
            "Production coverage %",
            "Import coverage %",
            "Domestic supply coverage %",
            "Food coverage %",
            "Per-capita quantity coverage %",
            "Calorie coverage %",
            "Minimum core coverage %",
            "Mean core coverage %",
            "Recent population-weighted kcal/capita/day",
            "Median country kcal/capita/day",
            "Countries with positive calories",
            "Imputed percentage",
            "Estimated percentage",
        ]
    ]
    .sort_values(
        "Recent population-weighted kcal/capita/day",
        ascending=False
    )
    .reset_index(drop=True)
)

display(candidate_selection_table)

,Item Code,Item Code (FBS),Item,Production coverage %,Import coverage %,Domestic supply coverage %,Food coverage %,Per-capita quantity coverage %,Calorie coverage %,Minimum core coverage %,Mean core coverage %,Recent population-weighted kcal/capita/day,Median country kcal/capita/day,Countries with positive calories,Imputed percentage,Estimated percentage
0,2514,'S2514,Maize and products,93.02,100.00,100.00,100.00,100.00,100.00,93.02,98.84,387.69,272.05,43.00,66.27,33.73
1,2511,'S2511,Wheat and products,62.46,100.00,100.00,100.00,100.00,100.00,62.46,93.74,364.51,225.92,43.00,64.44,35.56
2,2532,'S2532,Cassava and products,69.77,91.53,98.34,95.85,95.85,95.85,69.77,91.20,302.68,68.51,38.00,64.97,35.03
3,2807,'S2807,Rice and products,79.24,100.00,100.00,100.00,100.00,100.00,79.24,96.54,236.00,273.17,43.00,65.47,34.53
4,2518,'S2518,Sorghum and products,76.74,78.90,89.04,82.89,82.89,82.89,76.74,82.23,92.87,16.99,37.00,66.40,33.60
5,2535,'S2535,Yams,41.20,42.03,68.60,61.30,61.30,61.63,41.20,56.01,83.14,1.38,21.00,63.42,36.58
6,2549,'S2549,"Pulses, Other and products",87.38,99.34,99.34,99.34,99.34,99.34,87.38,97.34,62.30,29.54,43.00,65.98,34.02
7,2552,'S2552,Groundnuts,88.37,98.50,99.34,99.34,99.34,99.34,88.37,97.37,44.88,23.44,43.00,65.99,34.01
8,2616,'S2616,Plantains,38.87,57.81,82.89,78.57,78.57,78.57,38.87,69.21,44.11,0.76,28.00,62.16,37.84
9,2517,'S2517,Millet and products,67.44,83.22,92.36,76.41,76.41,76.41,67.44,78.71,42.82,5.81,34.00,67.64,32.36


In [42]:
candidate_code_strings = set(
    STAPLE_CANDIDATE_CODES
)

candidate_recent_calories = (
    recent_calories.loc[
        recent_calories[
            "Item Code"
        ]
        .astype("string")
        .isin(candidate_code_strings)
    ]
    .copy()
)

candidate_country_relevance = (
    candidate_recent_calories.groupby(
        ITEM_ID_COLUMNS
        + [
            "M49 Code",
            "Area",
        ],
        dropna=False
    )
    .agg(
        Recent_mean_kcal=(
            "Value",
            "mean"
        ),
        Recent_years_recorded=(
            "Year",
            "nunique"
        ),
    )
    .reset_index()
)

candidate_country_relevance[
    "Above 0 kcal"
] = (
    candidate_country_relevance[
        "Recent_mean_kcal"
    ].gt(0)
)

candidate_country_relevance[
    "At least 5 kcal"
] = (
    candidate_country_relevance[
        "Recent_mean_kcal"
    ].ge(5)
)

candidate_country_relevance[
    "At least 10 kcal"
] = (
    candidate_country_relevance[
        "Recent_mean_kcal"
    ].ge(10)
)

candidate_country_relevance[
    "At least 25 kcal"
] = (
    candidate_country_relevance[
        "Recent_mean_kcal"
    ].ge(25)
)

candidate_country_relevance[
    "At least 50 kcal"
] = (
    candidate_country_relevance[
        "Recent_mean_kcal"
    ].ge(50)
)

In [43]:
relevance_counts = (
    candidate_country_relevance.groupby(
        ITEM_ID_COLUMNS,
        dropna=False
    )
    .agg(
        Countries_with_recent_records=(
            "M49 Code",
            "nunique"
        ),
        Countries_with_all_3_recent_years=(
            "Recent_years_recorded",
            lambda values: (
                values.eq(3).sum()
            )
        ),
        Countries_above_0_kcal=(
            "Above 0 kcal",
            "sum"
        ),
        Countries_at_least_5_kcal=(
            "At least 5 kcal",
            "sum"
        ),
        Countries_at_least_10_kcal=(
            "At least 10 kcal",
            "sum"
        ),
        Countries_at_least_25_kcal=(
            "At least 25 kcal",
            "sum"
        ),
        Countries_at_least_50_kcal=(
            "At least 50 kcal",
            "sum"
        ),
    )
    .reset_index()
)

relevance_counts = (
    relevance_counts.merge(
        candidate_selection_table[
            [
                "Item Code",
                "Item Code (FBS)",
                "Item",
                "Recent population-weighted kcal/capita/day",
                "Median country kcal/capita/day",
            ]
        ],
        on=ITEM_ID_COLUMNS,
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "Recent population-weighted kcal/capita/day",
        ascending=False
    )
    .reset_index(drop=True)
)

display(relevance_counts)

,Item Code,Item Code (FBS),Item,Countries_with_recent_records,Countries_with_all_3_recent_years,Countries_above_0_kcal,Countries_at_least_5_kcal,Countries_at_least_10_kcal,Countries_at_least_25_kcal,Countries_at_least_50_kcal,Recent population-weighted kcal/capita/day,Median country kcal/capita/day
0,2514,'S2514,Maize and products,43,43,43,41,40,37,35,387.69,272.05
1,2511,'S2511,Wheat and products,43,43,43,43,43,43,39,364.51,225.92
2,2532,'S2532,Cassava and products,42,42,38,28,27,24,22,302.68,68.51
3,2807,'S2807,Rice and products,43,43,43,43,43,41,37,236.00,273.17
4,2518,'S2518,Sorghum and products,38,37,37,25,23,16,11,92.87,16.99
5,2535,'S2535,Yams,29,28,21,13,9,7,5,83.14,1.38
6,2549,'S2549,"Pulses, Other and products",43,43,43,34,29,23,14,62.30,29.54
7,2552,'S2552,Groundnuts,43,43,43,38,28,20,15,44.88,23.44
8,2616,'S2616,Plantains,41,36,28,15,15,12,9,44.11,0.76
9,2517,'S2517,Millet and products,36,35,34,21,17,11,8,42.82,5.81


In [44]:
RELEVANCE_THRESHOLD_KCAL = 5

relevant_country_item_pairs = (
    candidate_country_relevance.loc[
        candidate_country_relevance[
            "Recent_mean_kcal"
        ].ge(RELEVANCE_THRESHOLD_KCAL),
        ITEM_ID_COLUMNS
        + [
            "M49 Code",
            "Area",
        ]
    ]
    .drop_duplicates()
)

relevant_country_counts = (
    relevant_country_item_pairs.groupby(
        ITEM_ID_COLUMNS,
        dropna=False
    )["M49 Code"]
    .nunique()
    .rename("Relevant countries")
    .reset_index()
)

display(
    relevant_country_counts.sort_values(
        "Relevant countries",
        ascending=False
    )
)

,Item Code,Item Code (FBS),Item,Relevant countries
0,2511,'S2511,Wheat and products,43
13,2807,'S2807,Rice and products,43
1,2514,'S2514,Maize and products,41
10,2552,'S2552,Groundnuts,38
9,2549,'S2549,"Pulses, Other and products",34
4,2531,'S2531,Potatoes and products,32
5,2532,'S2532,Cassava and products,28
8,2546,'S2546,Beans,27
3,2518,'S2518,Sorghum and products,25
6,2533,'S2533,Sweet potatoes,25


In [45]:
relevant_core_history = (
    individual_commodity_long.loc[
        individual_commodity_long[
            "Element"
        ].isin(CORE_ELEMENTS)
    ]
    .merge(
        relevant_country_item_pairs[
            ITEM_ID_COLUMNS
            + ["M49 Code"]
        ],
        on=(
            ITEM_ID_COLUMNS
            + ["M49 Code"]
        ),
        how="inner",
        validate="many_to_one"
    )
)

relevant_core_counts = (
    relevant_core_history.loc[
        relevant_core_history[
            "Value"
        ].notna()
    ]
    .groupby(
        ITEM_ID_COLUMNS
        + ["Element"],
        dropna=False
    )
    .size()
    .rename(
        "Recorded relevant country-years"
    )
    .reset_index()
)

relevant_core_counts = (
    relevant_core_counts.merge(
        relevant_country_counts,
        on=ITEM_ID_COLUMNS,
        how="left",
        validate="many_to_one"
    )
)

relevant_core_counts[
    "Possible relevant country-years"
] = (
    relevant_core_counts[
        "Relevant countries"
    ]
    * NUMBER_OF_YEARS
)

relevant_core_counts[
    "Relevant coverage percentage"
] = (
    relevant_core_counts[
        "Recorded relevant country-years"
    ]
    / relevant_core_counts[
        "Possible relevant country-years"
    ]
    * 100
)

In [46]:
relevant_coverage_wide = (
    relevant_core_counts.pivot(
        index=ITEM_ID_COLUMNS,
        columns="Element",
        values="Relevant coverage percentage"
    )
    .reset_index()
)

relevant_coverage_wide.columns.name = None

relevant_coverage_names = {
    "Production":
        "Relevant production coverage %",
    "Import quantity":
        "Relevant import coverage %",
    "Domestic supply quantity":
        "Relevant domestic supply coverage %",
    "Food":
        "Relevant food coverage %",
    "Food supply quantity (kg/capita/yr)":
        "Relevant per-capita quantity coverage %",
    "Food supply (kcal/capita/day)":
        "Relevant calorie coverage %",
}

relevant_coverage_wide = (
    relevant_coverage_wide.rename(
        columns=relevant_coverage_names
    )
)

relevance_adjusted_evidence = (
    relevance_counts
    .merge(
        relevant_country_counts,
        on=ITEM_ID_COLUMNS,
        how="left",
        validate="one_to_one"
    )
    .merge(
        relevant_coverage_wide,
        on=ITEM_ID_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

relevance_adjusted_evidence = (
    relevance_adjusted_evidence[
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
            "Recent population-weighted kcal/capita/day",
            "Median country kcal/capita/day",
            "Countries_above_0_kcal",
            "Countries_at_least_5_kcal",
            "Countries_at_least_10_kcal",
            "Countries_at_least_25_kcal",
            "Countries_at_least_50_kcal",
            "Relevant countries",
            "Relevant production coverage %",
            "Relevant import coverage %",
            "Relevant domestic supply coverage %",
            "Relevant food coverage %",
            "Relevant per-capita quantity coverage %",
            "Relevant calorie coverage %",
        ]
    ]
    .sort_values(
        "Recent population-weighted kcal/capita/day",
        ascending=False
    )
    .reset_index(drop=True)
)

display(relevance_adjusted_evidence)

,Item Code,Item Code (FBS),Item,Recent population-weighted kcal/capita/day,Median country kcal/capita/day,Countries_above_0_kcal,Countries_at_least_5_kcal,Countries_at_least_10_kcal,Countries_at_least_25_kcal,Countries_at_least_50_kcal,Relevant countries,Relevant production coverage %,Relevant import coverage %,Relevant domestic supply coverage %,Relevant food coverage %,Relevant per-capita quantity coverage %,Relevant calorie coverage %
0,2514,'S2514,Maize and products,387.69,272.05,43,41,40,37,35,41,97.56,100.00,100.00,100.00,100.00,100.00
1,2511,'S2511,Wheat and products,364.51,225.92,43,43,43,43,39,43,62.46,100.00,100.00,100.00,100.00,100.00
2,2532,'S2532,Cassava and products,302.68,68.51,38,28,27,24,22,28,100.00,93.62,100.00,100.00,100.00,100.00
3,2807,'S2807,Rice and products,236.00,273.17,43,43,43,41,37,43,79.24,100.00,100.00,100.00,100.00,100.00
4,2518,'S2518,Sorghum and products,92.87,16.99,37,25,23,16,11,25,100.00,88.57,100.00,100.00,100.00,100.00
5,2535,'S2535,Yams,83.14,1.38,21,13,9,7,5,13,97.80,57.69,100.00,100.00,100.00,100.00
6,2549,'S2549,"Pulses, Other and products",62.30,29.54,43,34,29,23,14,34,94.12,100.00,100.00,100.00,100.00,100.00
7,2552,'S2552,Groundnuts,44.88,23.44,43,38,28,20,15,38,89.47,98.31,99.25,99.25,99.25,99.25
8,2616,'S2616,Plantains,44.11,0.76,28,15,15,12,9,15,98.10,51.43,98.10,98.10,98.10,98.10
9,2517,'S2517,Millet and products,42.82,5.81,34,21,17,11,8,21,100.00,86.39,100.00,100.00,100.00,100.00


In [47]:
selection_decisions = pd.DataFrame(
    [
        {
            "Item Code": "2514",
            "Commodity": "Maize and products",
            "Selection role": "Continental anchor",
            "Selection reason": (
                "Highest recent calorie importance; "
                "relevant across most covered countries"
            ),
        },
        {
            "Item Code": "2511",
            "Commodity": "Wheat and products",
            "Selection role": "Continental anchor",
            "Selection reason": (
                "High importance in all 43 countries; "
                "captures import-dependent supply risk"
            ),
        },
        {
            "Item Code": "2532",
            "Commodity": "Cassava and products",
            "Selection role": "Regional staple",
            "Selection reason": (
                "High calorie contribution and strong "
                "coverage in relevant countries"
            ),
        },
        {
            "Item Code": "2807",
            "Commodity": "Rice and products",
            "Selection role": "Continental anchor",
            "Selection reason": (
                "Important in all 43 countries with "
                "complete food-supply coverage"
            ),
        },
        {
            "Item Code": "2518",
            "Commodity": "Sorghum and products",
            "Selection role": "Regional staple",
            "Selection reason": (
                "Important dryland cereal with complete "
                "supply coverage in relevant countries"
            ),
        },
        {
            "Item Code": "2535",
            "Commodity": "Yams",
            "Selection role": "Regional staple",
            "Selection reason": (
                "High calorie importance in its core "
                "countries, especially West Africa"
            ),
        },
        {
            "Item Code": "2517",
            "Commodity": "Millet and products",
            "Selection role": "Regional staple",
            "Selection reason": (
                "Important dryland staple with complete "
                "supply coverage in relevant countries"
            ),
        },
        {
            "Item Code": "2552",
            "Commodity": "Groundnuts",
            "Selection role": "Protein and oilseed staple",
            "Selection reason": (
                "Broad relevance, strong coverage and "
                "nutritional diversification"
            ),
        },
    ]
)

SELECTED_ITEM_CODES = set(
    selection_decisions["Item Code"]
)

print(
    "Number of proposed commodities:",
    len(SELECTED_ITEM_CODES)
)

display(selection_decisions)

Number of proposed commodities: 8


,Item Code,Commodity,Selection role,Selection reason
0,2514,Maize and products,Continental anchor,Highest recent calorie importance; relevant ac...
1,2511,Wheat and products,Continental anchor,High importance in all 43 countries; captures ...
2,2532,Cassava and products,Regional staple,High calorie contribution and strong coverage ...
3,2807,Rice and products,Continental anchor,Important in all 43 countries with complete fo...
4,2518,Sorghum and products,Regional staple,Important dryland cereal with complete supply ...
5,2535,Yams,Regional staple,"High calorie importance in its core countries,..."
6,2517,Millet and products,Regional staple,Important dryland staple with complete supply ...
7,2552,Groundnuts,Protein and oilseed staple,"Broad relevance, strong coverage and nutrition..."


In [48]:
selected_identity_check = (
    item_identity.loc[
        item_identity[
            "Item Code"
        ]
        .astype("string")
        .isin(SELECTED_ITEM_CODES)
    ]
    .copy()
)

selected_identity_check[
    "Item Code"
] = (
    selected_identity_check[
        "Item Code"
    ]
    .astype("string")
)

selection_identity_validation = (
    selection_decisions.merge(
        selected_identity_check,
        on="Item Code",
        how="left",
        validate="one_to_one"
    )
)

selection_identity_validation[
    "Label matches source"
] = (
    selection_identity_validation[
        "Commodity"
    ]
    == selection_identity_validation[
        "Item"
    ]
)

display(
    selection_identity_validation[
        [
            "Item Code",
            "Item Code (FBS)",
            "Commodity",
            "Item",
            "Selection role",
            "Label matches source",
        ]
    ]
)

assert (
    selection_identity_validation[
        "Item"
    ].notna().all()
)

assert (
    selection_identity_validation[
        "Label matches source"
    ].all()
)

print(
    "\nAll proposed item identities "
    "match the raw FAO source."
)

,Item Code,Item Code (FBS),Commodity,Item,Selection role,Label matches source
0,2514,'S2514,Maize and products,Maize and products,Continental anchor,True
1,2511,'S2511,Wheat and products,Wheat and products,Continental anchor,True
2,2532,'S2532,Cassava and products,Cassava and products,Regional staple,True
3,2807,'S2807,Rice and products,Rice and products,Continental anchor,True
4,2518,'S2518,Sorghum and products,Sorghum and products,Regional staple,True
5,2535,'S2535,Yams,Yams,Regional staple,True
6,2517,'S2517,Millet and products,Millet and products,Regional staple,True
7,2552,'S2552,Groundnuts,Groundnuts,Protein and oilseed staple,True



All proposed item identities match the raw FAO source.


In [50]:
# Create the selected commodity subset

selected_commodities_long = (
    individual_commodity_long.loc[
        individual_commodity_long["Item Code"]
        .astype("string")
        .isin(SELECTED_ITEM_CODES)
    ]
    .copy()
    .reset_index(drop=True)
)

selected_commodities_long["Item Code numeric"] = (
    pd.to_numeric(
        selected_commodities_long["Item Code"],
        errors="coerce"
    )
)

selected_items_found = set(
    selected_commodities_long["Item Code"]
    .astype("string")
    .unique()
)

assert selected_items_found == SELECTED_ITEM_CODES

assert not selected_commodities_long[
    "Item Code numeric"
].between(2900, 2999).any()

assert not selected_commodities_long[
    "Element"
].eq(POPULATION_ELEMENT).any()

print(
    "Selected commodity subset "
    "created successfully."
)

Selected commodity subset created successfully.


In [51]:
ITEM_SUMMARY_KEYS = [
    "Item Code",
    "Item Code (FBS)",
    "Item",
]

selected_item_base_summary = (
    selected_commodities_long.groupby(
        ITEM_SUMMARY_KEYS,
        dropna=False
    )
    .agg(
        Source_rows=("Value", "size"),
        Recorded_values=("Value", "count"),
        Countries=("M49 Code", "nunique"),
        Elements=("Element", "nunique"),
        Years=("Year", "nunique"),
    )
    .reset_index()
)

selected_recorded_quality = (
    selected_commodities_long.loc[
        selected_commodities_long[
            "Value"
        ].notna()
    ]
    .copy()
)

selected_recorded_quality["Is_imputed"] = (
    selected_recorded_quality[
        "Flag"
    ].eq("I")
)

selected_recorded_quality["Is_estimated"] = (
    selected_recorded_quality[
        "Flag"
    ].eq("E")
)

selected_quality_summary = (
    selected_recorded_quality.groupby(
        ITEM_SUMMARY_KEYS,
        dropna=False
    )
    .agg(
        Imputed_share=(
            "Is_imputed",
            "mean"
        ),
        Estimated_share=(
            "Is_estimated",
            "mean"
        ),
    )
    .reset_index()
)

selected_item_summary = (
    selected_item_base_summary.merge(
        selected_quality_summary,
        on=ITEM_SUMMARY_KEYS,
        how="left",
        validate="one_to_one"
    )
)

selected_item_summary["Missing values"] = (
    selected_item_summary["Source_rows"]
    - selected_item_summary["Recorded_values"]
)

selected_item_summary["Recorded coverage %"] = (
    selected_item_summary["Recorded_values"]
    / selected_item_summary["Source_rows"]
    * 100
)

selected_item_summary["Imputed %"] = (
    selected_item_summary["Imputed_share"]
    * 100
)

selected_item_summary["Estimated %"] = (
    selected_item_summary["Estimated_share"]
    * 100
)

selected_item_summary = (
    selected_item_summary.drop(
        columns=[
            "Imputed_share",
            "Estimated_share",
        ]
    )
    .sort_values("Item")
    .reset_index(drop=True)
)

display(selected_item_summary)

,Item Code,Item Code (FBS),Item,Source_rows,Recorded_values,Countries,Elements,Years,Missing values,Recorded coverage %,Imputed %,Estimated %
0,2532,'S2532,Cassava and products,9562,8689,43,20,14,873,90.87,73.44,26.56
1,2552,'S2552,Groundnuts,10024,9785,43,20,14,239,97.62,75.55,24.45
2,2514,'S2514,Maize and products,11116,10802,43,20,14,314,97.18,77.71,22.29
3,2517,'S2517,Millet and products,9044,8061,43,20,14,983,89.13,77.17,22.83
4,2807,'S2807,Rice and products,10794,10440,43,20,14,354,96.72,76.93,23.07
5,2518,'S2518,Sorghum and products,9604,8623,43,20,14,981,89.79,76.85,23.15
6,2511,'S2511,Wheat and products,10528,10155,43,20,14,373,96.46,76.29,23.71
7,2535,'S2535,Yams,7322,5440,42,19,14,1882,74.30,72.76,27.24


In [52]:
selected_subset_summary = pd.Series(
    {
        "Rows": len(
            selected_commodities_long
        ),
        "Recorded values": (
            selected_commodities_long[
                "Value"
            ]
            .notna()
            .sum()
        ),
        "Missing values": (
            selected_commodities_long[
                "Value"
            ]
            .isna()
            .sum()
        ),
        "Countries represented": (
            selected_commodities_long[
                "M49 Code"
            ]
            .nunique()
        ),
        "Selected commodities": (
            selected_commodities_long[
                "Item Code"
            ]
            .nunique()
        ),
        "Elements": (
            selected_commodities_long[
                "Element"
            ]
            .nunique()
        ),
        "Years": (
            selected_commodities_long[
                "Year"
            ]
            .nunique()
        ),
        "Aggregate rows": (
            selected_commodities_long[
                "Item Code numeric"
            ]
            .between(2900, 2999)
            .sum()
        ),
        "Population rows": (
            selected_commodities_long[
                "Element"
            ]
            .eq(POPULATION_ELEMENT)
            .sum()
        ),
        "Values without flags": (
            (
                selected_commodities_long[
                    "Value"
                ].notna()
                & selected_commodities_long[
                    "Flag"
                ].isna()
            )
            .sum()
        ),
        "Flags without values": (
            (
                selected_commodities_long[
                    "Value"
                ].isna()
                & selected_commodities_long[
                    "Flag"
                ].notna()
            )
            .sum()
        ),
    },
    name="Result"
)

display(
    selected_subset_summary.to_frame()
)

,Result
Rows,77994
Recorded values,71995
Missing values,5999
Countries represented,43
Selected commodities,8
Elements,20
Years,14
Aggregate rows,0
Population rows,0
Values without flags,0


In [53]:
FINAL_SOURCE_COLUMNS = [
    "Area Code",
    "Area Code (M49)",
    "M49 Code",
    "Area",
    "Item Code",
    "Item Code (FBS)",
    "Item",
    "Element Code",
    "Element",
    "Unit",
    "Year",
    "Value",
    "Flag",
]

africa_selected_output = (
    selected_commodities_long[
        FINAL_SOURCE_COLUMNS
    ]
    .copy()
    .sort_values(
        [
            "Area",
            "Item Code",
            "Element Code",
            "Year",
        ]
    )
    .reset_index(drop=True)
)

display(africa_selected_output.head())

,Area Code,Area Code (M49),M49 Code,Area,Item Code,Item Code (FBS),Item,Element Code,Element,Unit,Year,Value,Flag
0,4,'012,012,Algeria,2511,'S2511,Wheat and products,645,Food supply quantity (kg/capita/yr),kg/cap,2010,180.60,E
1,4,'012,012,Algeria,2511,'S2511,Wheat and products,645,Food supply quantity (kg/capita/yr),kg/cap,2011,176.75,E
2,4,'012,012,Algeria,2511,'S2511,Wheat and products,645,Food supply quantity (kg/capita/yr),kg/cap,2012,180.26,E
3,4,'012,012,Algeria,2511,'S2511,Wheat and products,645,Food supply quantity (kg/capita/yr),kg/cap,2013,174.50,E
4,4,'012,012,Algeria,2511,'S2511,Wheat and products,645,Food supply quantity (kg/capita/yr),kg/cap,2014,171.92,E


In [54]:
africa_population_output = (
    population_long[
        [
            "Area Code",
            "Area Code (M49)",
            "M49 Code",
            "Area",
            "Year",
            "Value",
            "Unit",
            "Flag",
        ]
    ]
    .rename(
        columns={
            "Value": "Population 1000",
            "Unit": "Population unit",
            "Flag": "Population flag",
        }
    )
    .copy()
)

africa_population_output[
    "Population"
] = (
    africa_population_output[
        "Population 1000"
    ]
    * 1000
)

africa_population_output = (
    africa_population_output.sort_values(
        [
            "Area",
            "Year",
        ]
    )
    .reset_index(drop=True)
)

display(africa_population_output.head())

,Area Code,Area Code (M49),M49 Code,Area,Year,Population 1000,Population unit,Population flag,Population
0,4,'012,012,Algeria,2010,"36,188.24",1000 No,X,"36,188,240.00"
1,4,'012,012,Algeria,2011,"36,903.38",1000 No,X,"36,903,380.00"
2,4,'012,012,Algeria,2012,"37,646.17",1000 No,X,"37,646,170.00"
3,4,'012,012,Algeria,2013,"38,414.17",1000 No,X,"38,414,170.00"
4,4,'012,012,Algeria,2014,"39,205.03",1000 No,X,"39,205,030.00"


In [55]:
selection_roles = {
    "2514": "Continental anchor",
    "2511": "Continental anchor",
    "2532": "Regional staple",
    "2807": "Continental anchor",
    "2518": "Regional staple",
    "2535": "Regional staple",
    "2517": "Regional staple",
    "2552": "Protein and oilseed staple",
}

selection_reasons = {
    "2514": (
        "Highest recent calorie importance; "
        "relevant across 41 countries"
    ),
    "2511": (
        "High importance in all 43 countries; "
        "captures import dependency"
    ),
    "2532": (
        "High calorie contribution and complete "
        "supply coverage in relevant countries"
    ),
    "2807": (
        "Important in all 43 countries with "
        "complete food-supply coverage"
    ),
    "2518": (
        "Important dryland cereal with complete "
        "supply coverage in relevant countries"
    ),
    "2535": (
        "High importance in its core countries; "
        "captures a distinct West African staple"
    ),
    "2517": (
        "Important dryland staple with complete "
        "supply coverage in relevant countries"
    ),
    "2552": (
        "Broad relevance, strong coverage and "
        "nutritional importance"
    ),
}

final_selection_metadata = (
    africa_selected_output[
        [
            "Item Code",
            "Item Code (FBS)",
            "Item",
        ]
    ]
    .drop_duplicates()
    .copy()
)

final_selection_metadata[
    "Item Code"
] = (
    final_selection_metadata[
        "Item Code"
    ]
    .astype("string")
)

final_selection_metadata[
    "Selection role"
] = (
    final_selection_metadata[
        "Item Code"
    ]
    .map(selection_roles)
)

final_selection_metadata[
    "Selection reason"
] = (
    final_selection_metadata[
        "Item Code"
    ]
    .map(selection_reasons)
)

assert final_selection_metadata[
    "Selection role"
].notna().all()

assert final_selection_metadata[
    "Selection reason"
].notna().all()

final_selection_metadata = (
    final_selection_metadata.sort_values(
        "Item"
    )
    .reset_index(drop=True)
)

display(final_selection_metadata)

,Item Code,Item Code (FBS),Item,Selection role,Selection reason
0,2532,'S2532,Cassava and products,Regional staple,High calorie contribution and complete supply ...
1,2552,'S2552,Groundnuts,Protein and oilseed staple,"Broad relevance, strong coverage and nutrition..."
2,2514,'S2514,Maize and products,Continental anchor,Highest recent calorie importance; relevant ac...
3,2517,'S2517,Millet and products,Regional staple,Important dryland staple with complete supply ...
4,2807,'S2807,Rice and products,Continental anchor,Important in all 43 countries with complete fo...
5,2518,'S2518,Sorghum and products,Regional staple,Important dryland cereal with complete supply ...
6,2511,'S2511,Wheat and products,Continental anchor,High importance in all 43 countries; captures ...
7,2535,'S2535,Yams,Regional staple,High importance in its core countries; capture...


In [56]:
SELECTED_PARQUET_PATH = (
    AFRICA_OUTPUT_DIR
    / "africa_selected_commodities_long_2010_2023.parquet"
)

POPULATION_PARQUET_PATH = (
    AFRICA_OUTPUT_DIR
    / "africa_population_2010_2023.parquet"
)

SELECTION_METADATA_PATH = (
    AFRICA_OUTPUT_DIR
    / "selected_commodity_metadata.csv"
)

SELECTION_EVIDENCE_PATH = (
    AFRICA_OUTPUT_DIR
    / "candidate_selection_evidence.csv"
)

COUNTRY_SCOPE_PATH = (
    AFRICA_OUTPUT_DIR
    / "africa_country_scope.csv"
)

africa_selected_output.to_parquet(
    SELECTED_PARQUET_PATH,
    engine="pyarrow",
    compression="snappy",
    index=False
)

africa_population_output.to_parquet(
    POPULATION_PARQUET_PATH,
    engine="pyarrow",
    compression="snappy",
    index=False
)

final_selection_metadata.to_csv(
    SELECTION_METADATA_PATH,
    index=False
)

relevance_adjusted_evidence.to_csv(
    SELECTION_EVIDENCE_PATH,
    index=False
)

africa_area_review.to_csv(
    COUNTRY_SCOPE_PATH,
    index=False
)

print("Africa-first files saved.")

Africa-first files saved.


In [57]:
saved_selected = pd.read_parquet(
    SELECTED_PARQUET_PATH
)

saved_population = pd.read_parquet(
    POPULATION_PARQUET_PATH
)

saved_item_codes = set(
    saved_selected["Item Code"]
    .astype("string")
    .unique()
)

readback_summary = pd.Series(
    {
        "Selected rows saved": (
            len(saved_selected)
        ),
        "Selected rows expected": (
            len(africa_selected_output)
        ),
        "Population rows saved": (
            len(saved_population)
        ),
        "Population rows expected": 602,
        "Selected commodities": (
            saved_selected[
                "Item Code"
            ].nunique()
        ),
        "Countries in selected data": (
            saved_selected[
                "M49 Code"
            ].nunique()
        ),
        "Countries in population data": (
            saved_population[
                "M49 Code"
            ].nunique()
        ),
        "Years in selected data": (
            saved_selected[
                "Year"
            ].nunique()
        ),
        "Years in population data": (
            saved_population[
                "Year"
            ].nunique()
        ),
        "Missing selected values": (
            saved_selected[
                "Value"
            ].isna().sum()
        ),
        "Duplicate population keys": (
            saved_population.duplicated(
                subset=[
                    "M49 Code",
                    "Year",
                ]
            ).sum()
        ),
    },
    name="Result"
)

display(readback_summary.to_frame())

assert len(saved_selected) == len(
    africa_selected_output
)

assert len(saved_population) == 602

assert saved_item_codes == (
    SELECTED_ITEM_CODES
)

assert saved_population.duplicated(
    subset=[
        "M49 Code",
        "Year",
    ]
).sum() == 0

assert not saved_selected[
    "Item"
].eq("Grand Total").any()

assert not saved_selected[
    "Element"
].eq(POPULATION_ELEMENT).any()

print(
    "\nAll saved-file validations passed."
)

,Result
Selected rows saved,77994
Selected rows expected,77994
Population rows saved,602
Population rows expected,602
Selected commodities,8
Countries in selected data,43
Countries in population data,43
Years in selected data,14
Years in population data,14
Missing selected values,5999



All saved-file validations passed.


In [58]:
saved_paths = [
    SELECTED_PARQUET_PATH,
    POPULATION_PARQUET_PATH,
    SELECTION_METADATA_PATH,
    SELECTION_EVIDENCE_PATH,
    COUNTRY_SCOPE_PATH,
]

for file_path in saved_paths:
    size_mb = (
        file_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{file_path.name}: "
        f"{size_mb:,.3f} MB"
    )
    print(f"  {file_path}")

africa_selected_commodities_long_2010_2023.parquet: 0.238 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/africa_selected_commodities_long_2010_2023.parquet
africa_population_2010_2023.parquet: 0.016 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/africa_population_2010_2023.parquet
selected_commodity_metadata.csv: 0.001 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/selected_commodity_metadata.csv
candidate_selection_evidence.csv: 0.002 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/candidate_selection_evidence.csv
africa_country_scope.csv: 0.004 MB
  /Users/adewale/Documents/food_security_predictor/data/processed/africa_first/africa_country_scope.csv
